# Проверка информативности сообщений через Qwen

Ноутбук оценивает сообщения по отдельности с ближайшим контекстом. Короткие ответы вроде «да», «ок» или эмодзи можно отправить на проверку, если сами по себе они не несут полезного содержания. Сообщения с вложениями остаются в корпусе.

**Ничего не удаляется автоматически.** Сохраняются исходные данные, оценка и объяснение Qwen. Записи с меткой `review` выносятся в отдельный файл; после просмотра можно вручную решить, что исключать из плоского текста.

In [ ]:
from pathlib import Path
import json, time, requests
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "data").is_dir()
)
INPUT_DIR = PROJECT_ROOT / "data/processed"
OUTPUT_DIR = PROJECT_ROOT / "data/processed/qwen_review"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BASE_URL = "http://localhost:11434"
MODEL_NAME = "qwen3.5:4b"
EXPORT_ROOTS = {
    "chatexport_2026-10-05_messages_from_2026-07.csv": "ChatExport_2026-10-05",
    "chatexport_2026-10-05_1_messages_from_2026-07.csv": "ChatExport_2026-10-05 (1)",
    "chatexport_2026-10-05_2_messages_from_2026-07.csv": "ChatExport_2026-10-05 (2)",
}
BATCH_SIZE = 12
TIMEOUT_SECONDS = 240
INPUT_FILES = sorted(INPUT_DIR.glob("*_messages_from_2026-07.csv"))
if not INPUT_FILES:
    raise FileNotFoundError(f"В {INPUT_DIR} не найдены CSV после июля 2026 года.")
print(f"Файлов: {len(INPUT_FILES)}")
for f in INPUT_FILES:
    print(f"- {f.name}: {len(pd.read_csv(f))} строк")

## Загрузка и контекст

Каждая запись получает стабильный `record_id`. Для оценки модели передаются текст и соседние сообщения в том же чате; полный датасет остаётся без изменений.

In [ ]:
frames = []
for file in INPUT_FILES:
    frame = pd.read_csv(file, keep_default_na=False)
    frame["input_file"] = file.name
    frames.append(frame)
data = pd.concat(frames, ignore_index=True)
data["record_id"] = (
    data["input_file"].astype(str) + "::" + data["message_id"].astype(str)
)
data["date_sort"] = pd.to_datetime(data["date"], errors="coerce", utc=True)
data = data.sort_values(
    ["chat_name", "date_sort", "message_id"], kind="stable"
).reset_index(drop=True)
data["prev_text"] = data.groupby("chat_name")["text"].shift(1).fillna("")
data["next_text"] = data.groupby("chat_name")["text"].shift(-1).fillna("")
data["has_file"] = data["file_paths"].apply(lambda x: bool(x and x != "[]"))
data["has_text"] = data["text"].astype(str).str.strip().ne("")
print(
    f"Всего записей: {len(data):,}; с текстом: {data.has_text.sum():,}; с файлами: {data.has_file.sum():,}"
)
display(data[["record_id", "chat_name", "date", "sender", "text", "has_file"]].head(8))

## Критерии оценки

`keep` — сообщение добавляет смысл, факт, вопрос, инструкцию, ссылку, важную эмоциональную или разговорную информацию либо содержит вложение. `review` — текст сам по себе почти не добавляет содержания (например, пустая реплика, технический шум, одиночная реакция или дублирование), с учётом соседних реплик. При сомнении выбирается `keep`. Это предварительная сортировка для человека, не окончательная очистка.

In [ ]:
SYSTEM_PROMPT = (
    "Ты помогаешь подготовить архив переписки для поиска и последующего чтения. "
    "Оценивай информативность целевой реплики, используя соседние сообщения как контекст. "
    'Верни JSON-объект вида {"items":[{"record_id":"точный id","label":"keep или review","reason":"краткая причина по-русски"}]}. '
    "Для каждого входного id ровно один результат. "
    "keep: есть самостоятельный смысл или важное контекстное значение (вопрос, ответ, факт, ссылка, договорённость, инструкция, личный опыт, осмысленная реакция); либо есть вложение/файл. При сомнении выбирай keep. "
    "review: пусто, случайный набор символов, только незначимый шум, очевидный дубль или реплика без полезного содержания даже с учётом контекста. "
    "Короткое сообщение не означает автоматически review. Не считай эмоциональную реакцию или короткий ответ бесполезным без учёта соседей. "
    "Причина не длиннее 5 слов. Не придумывай содержание вложений. Не меняй и не пересказывай текст."
)


def request_batch(rows):
    payload = []
    for row in rows:
        payload.append(
            {
                "record_id": row.record_id,
                "text": row.text,
                "sender": row.sender,
                "has_file": bool(row.has_file),
                "content_type": row.content_type,
                "previous_message": row.prev_text,
                "next_message": row.next_text,
            }
        )
    user_prompt = (
        "Оцени каждую целевую реплику. Ответь только JSON с items.\n"
        + json.dumps(payload, ensure_ascii=False)
    )
    response = requests.post(
        f"{BASE_URL}/api/chat",
        timeout=TIMEOUT_SECONDS,
        json={
            "model": MODEL_NAME,
            "stream": False,
            "think": False,
            "format": "json",
            "options": {"temperature": 0, "num_predict": 3072},
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
        },
    )
    response.raise_for_status()
    result = json.loads(response.json()["message"]["content"])
    return result["items"]


def classify_batch_safely(batch):
    try:
        answer = request_batch(batch)
        by_id = {str(x.get("record_id")): x for x in answer if isinstance(x, dict)}
        expected = {str(r.record_id) for r in batch}
        if set(by_id) != expected:
            raise ValueError("Qwen вернул неполный список")
        for item in by_id.values():
            if item.get("label") not in {"keep", "review"}:
                raise ValueError("Qwen вернул неизвестную метку")
        return by_id
    except Exception as exc:
        if len(batch) > 1:
            middle = len(batch) // 2
            return {
                **classify_batch_safely(batch[:middle]),
                **classify_batch_safely(batch[middle:]),
            }
        # Безопасное поведение при ошибке: запись остаётся, а сбой явно виден при просмотре.
        row = batch[0]
        return {
            str(row.record_id): {
                "record_id": str(row.record_id),
                "label": "keep",
                "reason": f"Оставлено: ошибка Qwen ({type(exc).__name__}).",
            }
        }


def classify_all(data, batch_size=BATCH_SIZE, checkpoint=True):
    cache_path = OUTPUT_DIR / "qwen_classification_checkpoint.csv"
    existing = {}
    if cache_path.exists():
        old = pd.read_csv(cache_path, keep_default_na=False)
        existing = {
            r.record_id: {"label": r.qwen_label, "reason": r.qwen_reason}
            for r in old.itertuples()
            if r.qwen_label in {"keep", "review"}
        }
    # Вложения сохраняем по умолчанию и не отправляем их в модель.
    for row in data.loc[data.has_file].itertuples():
        existing[row.record_id] = {
            "label": "keep",
            "reason": "Есть вложение; сохранено без фильтрации.",
        }
    todo = data.loc[(~data.record_id.isin(existing)) & (~data.has_file)].copy()
    for start in range(0, len(todo), batch_size):
        batch = list(todo.iloc[start : start + batch_size].itertuples(index=False))
        by_id = classify_batch_safely(batch)
        for row in batch:
            item = by_id[str(row.record_id)]
            label = item.get("label")
            if row.has_file and label == "review":
                label = "keep"
                reason = "Есть вложение; оставлено в корпусе для ручной проверки."
            else:
                reason = str(item.get("reason", "")).strip()[:300]
            existing[str(row.record_id)] = {"label": label, "reason": reason}
        if checkpoint:
            pd.DataFrame(
                [
                    {
                        "record_id": k,
                        "qwen_label": v["label"],
                        "qwen_reason": v["reason"],
                    }
                    for k, v in existing.items()
                ]
            ).to_csv(cache_path, index=False, encoding="utf-8-sig")
        print(f"Оценено: {len(existing):,}/{len(data):,}")
        time.sleep(0.05)
    scores = pd.DataFrame(
        [
            {"record_id": k, "qwen_label": v["label"], "qwen_reason": v["reason"]}
            for k, v in existing.items()
        ]
    )
    return data.merge(scores, on="record_id", how="left", validate="one_to_one")


# После прерывания классификация продолжится по checkpoint; сохранённые результаты не запрашиваются повторно.

## Запуск и результаты

Запустите следующую ячейку, чтобы Qwen пометил записи и сохранил прогресс после каждой группы. Будут созданы полный набор с оценками, файл записей `review` и сводка. Исходные CSV не перезаписываются.

In [ ]:
scored = classify_all(data)
full_path = OUTPUT_DIR / "all_messages_with_qwen_labels.csv"
# Preserve human decisions if this export cell is run again.
if full_path.exists():
    previous = pd.read_csv(full_path, keep_default_na=False)
    if {"record_id", "reviewer_decision"}.issubset(previous.columns):
        decisions = previous.set_index("record_id")["reviewer_decision"].to_dict()
        scored["reviewer_decision"] = scored.record_id.map(decisions).fillna("")
    else:
        scored["reviewer_decision"] = ""
else:
    scored["reviewer_decision"] = ""
review_path = OUTPUT_DIR / "messages_for_manual_review.csv"
keep_path = OUTPUT_DIR / "messages_keep_for_flat_text.csv"
summary_path = OUTPUT_DIR / "qwen_review_summary.csv"
attachments_path = OUTPUT_DIR / "attachments_inventory.csv"
scored.drop(columns=["date_sort"]).to_csv(full_path, index=False, encoding="utf-8-sig")
decisions = scored.reviewer_decision.fillna("").str.strip().str.lower()
invalid_decisions = ~decisions.isin(["", "keep", "exclude"])
if invalid_decisions.any():
    raise ValueError(
        "reviewer_decision принимает только keep, exclude или пустое значение."
    )

review = scored.loc[scored.qwen_label.eq("review") & decisions.eq("")].copy()
review.drop(columns=["date_sort"]).to_csv(
    review_path, index=False, encoding="utf-8-sig"
)
keep_by_default = scored.qwen_label.eq("keep") | scored.has_file
keep_mask = decisions.eq("keep") | (decisions.eq("") & keep_by_default)
keep = scored.loc[keep_mask].copy()
keep.drop(columns=["date_sort"]).to_csv(keep_path, index=False, encoding="utf-8-sig")
summary = (
    scored.groupby(["input_file", "chat_name", "qwen_label"], dropna=False)
    .size()
    .rename("messages")
    .reset_index()
)
summary.to_csv(summary_path, index=False, encoding="utf-8-sig")
attachment_rows = []
for row in keep.itertuples():
    for file_path in json.loads(row.file_paths or "[]"):
        export_root = PROJECT_ROOT / "data" / EXPORT_ROOTS[row.input_file]
        absolute = export_root / file_path
        attachment_rows.append(
            {
                "record_id": row.record_id,
                "chat_name": row.chat_name,
                "date": row.date,
                "sender": row.sender,
                "message_text": row.text,
                "file_path": file_path,
                "file_type": Path(file_path).suffix.lower(),
                "file_exists": absolute.is_file(),
            }
        )
attachments = pd.DataFrame(
    attachment_rows,
    columns=[
        "record_id",
        "chat_name",
        "date",
        "sender",
        "message_text",
        "file_path",
        "file_type",
        "file_exists",
    ],
)
attachments.to_csv(attachments_path, index=False, encoding="utf-8-sig")
print(f"Полный набор: {full_path} ({len(scored):,})")
print(f"На ручную проверку: {review_path} ({len(review):,})")
print(f"Для плоского текста (keep + вложения): {keep_path} ({len(keep):,})")
print(f"Реестр вложений: {attachments_path} ({len(attachments):,} путей)")
print(f"Сводка: {summary_path}")
display(summary)

## Просмотр кандидатов на исключение

Посмотрите распределение причин и примеры. Для проверки доступны исходный текст и соседние сообщения. Полный CSV с оценками можно дополнить решением человека; только после этого следует собирать очищенный текстовый корпус.

In [ ]:
if "scored" not in globals():
    full_path = OUTPUT_DIR / "all_messages_with_qwen_labels.csv"
    if full_path.exists():
        scored = pd.read_csv(full_path, keep_default_na=False)
    else:
        raise RuntimeError("Сначала выполните ячейку с запуском классификации.")
review = scored.loc[scored.qwen_label.eq("review")].copy()
print(f"Кандидатов на проверку: {len(review):,} из {len(scored):,}")
display(
    review[
        [
            "record_id",
            "chat_name",
            "date",
            "sender",
            "prev_text",
            "text",
            "next_text",
            "qwen_reason",
            "file_paths",
        ]
    ].head(100)
)
display(review["qwen_reason"].value_counts().rename_axis("reason").to_frame("count"))

## OCR вложений и сборка хронологической ленты

Видеофайлы, стикеры и прочие типы вложений исключены. В PDF берётся готовый текстовый слой там, где он есть; сканированные страницы и JPEG распознаются Tesseract. У каждой записи сохраняются дата, чат и автор; текст вложения ставится сразу после текста сообщения и получает имя исходного файла. На выходе сохраняются построчный CSV и единый TXT в общей хронологии.

Для PDF с заведомо испорченным OCR можно задать проверенное краткое описание в `MANUAL_ATTACHMENT_SUMMARIES`; сырой OCR остаётся в техническом архиве распознавания, а в поисковую ленту вместо него попадёт резюме и ссылка на оригинал.

In [ ]:
import re
import shutil
import subprocess
import tempfile

TESSERACT = shutil.which("tesseract")
PDFTOTEXT = shutil.which("pdftotext")
PDFTOPPM = shutil.which("pdftoppm")
TESSDATA_DIR = PROJECT_ROOT / "data/processed/ocr/tessdata_fast"
OCR_DIR = OUTPUT_DIR / "ocr"
OCR_DIR.mkdir(parents=True, exist_ok=True)
assert TESSERACT, "Не найден Tesseract OCR (установите tesseract). "
assert PDFTOTEXT and PDFTOPPM, "Не найден Poppler (pdftotext и pdftoppm)."
assert (TESSDATA_DIR / "rus.traineddata").exists(), (
    "Не найдена русская модель Tesseract."
)

keep_for_ocr = pd.read_csv(
    OUTPUT_DIR / "messages_keep_for_flat_text.csv", keep_default_na=False
)
inv = pd.read_csv(OUTPUT_DIR / "attachments_inventory.csv", keep_default_na=False)
inv = inv.merge(keep_for_ocr[["record_id", "input_file"]], on="record_id", how="left")
inv = inv[
    inv.file_type.isin([".pdf", ".jpg", ".jpeg"])
    & inv.file_exists.astype(str).str.lower().eq("true")
].copy()
inv["file_id"] = inv["input_file"] + "::" + inv["file_path"]
# Один файл мог быть переслан в чат несколько раз: распознаём один раз.
unique_files = inv.drop_duplicates(["input_file", "file_path"])[
    ["input_file", "file_path"]
]
cache_path = OCR_DIR / "ocr_file_cache.csv"
cache = {}
if cache_path.exists():
    cached = pd.read_csv(cache_path, keep_default_na=False)
    cache = {r.file_id: r._asdict() for r in cached.itertuples(index=False)}


def run_tesseract(image_path):
    result = subprocess.run(
        [
            TESSERACT,
            str(image_path),
            "stdout",
            "-l",
            "rus+eng",
            "--tessdata-dir",
            str(TESSDATA_DIR),
            "--psm",
            "6",
        ],
        capture_output=True,
        text=True,
        errors="replace",
        check=True,
    )
    return result.stdout.strip()


def extract_pdf(path):
    extracted = subprocess.run(
        [PDFTOTEXT, "-layout", str(path), "-"],
        capture_output=True,
        text=True,
        errors="replace",
        check=True,
    ).stdout
    pages = extracted.split("\f")
    while pages and not pages[-1].strip():
        pages.pop()
    if not pages:
        pages = [""]
    output, methods = [], []
    with tempfile.TemporaryDirectory(prefix="pdf_ocr_") as tmp:
        for page_no, page_text in enumerate(pages, start=1):
            meaningful = len(re.sub(r"\s+", "", page_text)) >= 30
            if meaningful:
                output.append(page_text.strip())
                methods.append("pdf_text_layer")
                continue
            prefix = Path(tmp) / "page"
            subprocess.run(
                [
                    PDFTOPPM,
                    "-f",
                    str(page_no),
                    "-l",
                    str(page_no),
                    "-r",
                    "250",
                    "-png",
                    "-singlefile",
                    str(path),
                    str(prefix),
                ],
                capture_output=True,
                check=True,
            )
            image_path = Path(f"{prefix}.png")
            if image_path.exists():
                output.append(run_tesseract(image_path))
                methods.append("ocr")
            else:
                output.append("")
                methods.append("failed")
    used = set(methods)
    method = "mixed" if len(used) > 1 else (next(iter(used)) if used else "no_text")
    return "\n\n".join(t for t in output if t).strip(), method, len(pages)


for file_row in unique_files.itertuples(index=False):
    cache_id = file_row.input_file + "::" + file_row.file_path
    if cache_id in cache:
        continue
    root = PROJECT_ROOT / "data" / EXPORT_ROOTS[file_row.input_file]
    source = root / file_row.file_path
    try:
        if source.suffix.lower() == ".pdf":
            text, method, pages = extract_pdf(source)
        else:
            text = run_tesseract(source)
            method, pages = "ocr", 1
        record = {
            "file_id": cache_id,
            "input_file": file_row.input_file,
            "file_path": file_row.file_path,
            "file_type": source.suffix.lower(),
            "method": method,
            "pages": pages,
            "ocr_text": text,
            "status": "text_found" if text else "no_text_found",
            "error": "",
        }
    except Exception as exc:
        record = {
            "file_id": cache_id,
            "input_file": file_row.input_file,
            "file_path": file_row.file_path,
            "file_type": source.suffix.lower(),
            "method": "failed",
            "pages": "",
            "ocr_text": "",
            "status": "error",
            "error": f"{type(exc).__name__}: {exc}"[:300],
        }
    cache[cache_id] = record
    pd.DataFrame(cache.values()).to_csv(cache_path, index=False, encoding="utf-8-sig")
    print(f"{source.name}: {record['status']} ({record['method']})")

file_results = pd.DataFrame(cache.values())
file_results.to_csv(cache_path, index=False, encoding="utf-8-sig")
# Разворачиваем результаты на сообщения, сохраняя чат, сообщение и путь к первоисточнику.
ocr_rows = inv.merge(
    file_results,
    on=["file_id", "input_file", "file_path"],
    how="left",
    suffixes=("", "_ocr"),
)
ocr_rows = ocr_rows[
    [
        "record_id",
        "chat_name",
        "date",
        "sender",
        "message_text",
        "file_path",
        "file_type",
        "method",
        "pages",
        "status",
        "ocr_text",
        "error",
    ]
]
ocr_output = OCR_DIR / "ocr_attachment_text.csv"
ocr_rows.to_csv(ocr_output, index=False, encoding="utf-8-sig")

attachment_blocks = {}
MANUAL_ATTACHMENT_SUMMARIES = {
    (
        "chatexport_2026-10-05_messages_from_2026-07.csv::12851",
        "chats/chat_562952730570383/topic_6/files/---Т.в. 1-30.pdf",
    ): (
        "Контрольная работа по теории вероятностей («Хайзенберг»), 30-страничный скан. "
        "Автор сообщения представил PDF как контрольную работу по теории вероятностей. "
        "Текст OCR с этого скана ненадёжен, поэтому задания, условия и ответы по распознаванию не пересказываются. "
        "В предшествующей переписке обсуждаются экзамены; конкретные темы или задачи из PDF там не названы. "
        "Вскоре после загрузки в чате есть реплика «Ну что это такое..», но экспорт не подтверждает, "
        "что она относится именно к этому файлу."
    ),
}
for row in ocr_rows.itertuples(index=False):
    recognized = MANUAL_ATTACHMENT_SUMMARIES.get(
        (row.record_id, row.file_path), str(row.ocr_text or "").strip()
    )
    if recognized:
        attachment_blocks.setdefault(row.record_id, []).append(
            f"[Вложение: {Path(row.file_path).name}]\n{recognized}"
        )
flat = keep_for_ocr.copy()
flat["attachment_text"] = flat.record_id.map(
    lambda key: "\n\n".join(attachment_blocks.get(key, []))
)
files_by_record = ocr_rows.groupby("record_id").file_path.apply(list).to_dict()
flat["file_paths"] = flat.record_id.map(files_by_record).apply(
    lambda paths: json.dumps(
        paths if isinstance(paths, list) else [], ensure_ascii=False
    )
)
flat["flat_text"] = flat.apply(
    lambda r: "\n\n".join(
        x for x in [str(r.text).strip(), str(r.attachment_text).strip()] if x
    ),
    axis=1,
)
flat["date_sort"] = pd.to_datetime(flat["date"], errors="coerce", utc=True)
flat = flat.sort_values(
    ["date_sort", "chat_name", "message_id"], kind="stable"
).reset_index(drop=True)
with_text_path = OUTPUT_DIR / "messages_keep_with_attachment_text.csv"
flat.drop(columns=["date_sort"]).to_csv(
    with_text_path, index=False, encoding="utf-8-sig"
)
message_csv_path = OUTPUT_DIR / "messages_flat_text_with_ocr.csv"
flat[["record_id", "chat_name", "date", "sender", "flat_text", "file_paths"]].to_csv(
    message_csv_path, index=False, encoding="utf-8-sig"
)


def timeline_entry(row):
    author = str(row.sender).strip() or "Автор не указан"
    body = str(row.flat_text).strip()
    if not body:
        return ""
    path_links = []
    for rel_path in json.loads(row.file_paths or "[]"):
        absolute = (
            PROJECT_ROOT / "data" / EXPORT_ROOTS[row.input_file] / rel_path
        ).resolve()
        path_links.append(f"[Оригинал: {Path(rel_path).name}](<{absolute}>)")
    suffix = "\n" + "\n".join(path_links) if path_links else ""
    return f"[{row.date} | {row.chat_name} | {author}]\n{body}{suffix}"


timeline_path = OUTPUT_DIR / "chronological_corpus.txt"
entries = [timeline_entry(row) for row in flat.itertuples(index=False)]
entries = [entry for entry in entries if entry]
timeline_header = (
    "Хронологическая лента сообщений и текста вложений. "
    "Период: с 1 июля 2026 года; время в часовом поясе исходного экспорта.\n\n"
)
timeline_path.write_text(
    timeline_header + "\n\n".join(entries) + "\n", encoding="utf-8"
)
print(f"Вложений обработано: {len(file_results)}")
print(file_results.status.value_counts().to_dict())
print(f"Сообщений в хронологическом CSV: {len(flat)}")
print(f"Записей в TXT с текстом: {len(entries)}")
print(f"CSV: {message_csv_path}")
print(f"Единая лента: {timeline_path}")